# Beyond Sicilian: NLLB and our adapter on the other varieties of Italy

The adapter was trained on Sicilian (with English and Italian); this notebook measures what it
does to the neighbouring varieties that NLLB-200 supports, i.e., Friulian (`fur`), Ligurian
(`lij`), Lombard (`lmo`), Sardinian (`srd`) and Venetian (`vec`), with Sicilian as the reference.
Each is translated to and from English and Italian on the FLORES-200 devtest (1012
multi-parallel sentences), first by NLLB-1.3B zero-shot and then with the curriculum adapter of
§7.9 (`nllb-lora-curriculum-1.3B`). A gain on a variety means that the adaptation to Sicilian
transfers to it, a loss that the adapter forgets it (as it forgets it->en, cf. §7.4). Decoding
only; a T4 is enough. Results go to `results_varieties.json` on Drive; directions already
scored are skipped, so a disconnected run resumes. The companion LaBSE measurement (would our
aligner work on these varieties?) is `experiments/extraction/labse_varieties.py`.

In [ ]:
!pip -q install transformers datasets peft sentencepiece sacrebleu accelerate
!pip -q uninstall -y torchao
import os, sys, json
if not os.path.exists('SicilianNMT'):
    !git clone -q https://github.com/dmeoli/SicilianNMT.git
!git -C SicilianNMT pull -q
sys.path += ['SicilianNMT/experiments/nllb', 'SicilianNMT/experiments/dataset']
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/SicilianNMT-colab'
def read(p): return open(p, encoding='utf-8').read().splitlines()
from nllb_pipeline import LANG, load_base, translate, score
LANG.update({'fur': 'fur_Latn', 'lij': 'lij_Latn', 'lmo': 'lmo_Latn', 'srd': 'srd_Latn', 'vec': 'vec_Latn'})
if not os.path.exists('flores200_dataset'):
    os.system('wget -q https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz -O fl.tar.gz && tar -xzf fl.tar.gz')
fl = {l: read(f'flores200_dataset/devtest/{LANG[l]}.devtest') for l in ('en', 'it', 'scn', 'fur', 'lij', 'lmo', 'srd', 'vec')}
print('ready', {k: len(v) for k, v in fl.items()})

In [ ]:
VARIETIES = ('scn', 'fur', 'lij', 'lmo', 'srd', 'vec')
DIRS = [(x, p) for x in VARIETIES for p in ('en', 'it')] + [(p, x) for x in VARIETIES for p in ('en', 'it')]
path = f'{OUT}/results_varieties.json'
res = json.load(open(path)) if os.path.exists(path) else {}
for name, adapter in [('zero-shot', None), ('curriculum', f'{OUT}/nllb-lora-curriculum-1.3B')]:
    todo = [(s, t) for s, t in DIRS if f'{name} {s}->{t}' not in res]
    if not todo: continue
    m = None
    m, tok = load_base('facebook/nllb-200-1.3B', adapter=adapter)
    for s, t in todo:
        res[f'{name} {s}->{t}'] = score(translate(m, tok, fl[s], s, t), fl[t])
        json.dump(res, open(path, 'w'), indent=2)
        print(f'{name:10s} {s}->{t}: BLEU={res[f"{name} {s}->{t}"][0]} chrF={res[f"{name} {s}->{t}"][1]}')
print(f"{'direction':12s} {'zero-shot':>10s} {'curriculum':>11s} {'delta':>7s}")
for s, t in DIRS:
    z, c = res.get(f'zero-shot {s}->{t}'), res.get(f'curriculum {s}->{t}')
    if z and c: print(f"{s+'->'+t:12s} {z[0]:>10} {c[0]:>11} {c[0] - z[0]:>+7.2f}")